# TorGen v2: Analysis Notebook

Diagnostic plots comparing generated tornado tracks against ground truth.

**Runtime:** Colab CPU (no GPU required).  
**Data:** `samples.parquet` (generated) + `.pt.gz` files (GT + weather) on Google Drive.

In [ ]:
!pip install -q git+https://github.com/jcaramichaellehigh/TorGen.git cartopy
!pip show torgen | grep -E "^(Version|Location)"

In [ ]:
import os
import gzip
import io
import math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import torch
import pyproj
import cartopy.io.shapereader as shpreader
from scipy import stats

from torgen.data.dataset import (
    _load_pt,
    coords_to_bearing_length,
    bearing_length_to_coords,
    SPLIT_YEARS,
)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# --- Paths (edit these) ---
RAW_DIR = "/content/drive/MyDrive/raw"          # .pt.gz files
SAMPLES_PATH = "/content/drive/MyDrive/checkpoints/eval/samples.parquet"

# --- NARR projection constants ---
NARR_PROJ = (
    "+proj=lcc +lon_0=-107 +lat_0=50 +x_0=5632730 +y_0=4612610 "
    "+lat_1=50 +lat_2=50 +ellps=sphere"
)
EASTING_MIN, EASTING_MAX = 5_323_932.0, 8_213_139.0
NORTHING_MIN, NORTHING_MAX = 1_817_928.0, 4_707_135.0

STP_CHANNEL = 8
EXISTS_THRESHOLD = 0.5
SPLIT = "test"

In [ ]:
# --- State boundary helper ---
_STATE_CACHE = None

def _draw_states(ax, color="gray", linewidth=0.5, alpha=0.6):
    """Draw US state boundaries in [0,1] normalized NARR space."""
    global _STATE_CACHE
    if _STATE_CACHE is None:
        proj = pyproj.Proj(NARR_PROJ)
        shpfile = shpreader.natural_earth(
            resolution="110m", category="cultural",
            name="admin_1_states_provinces_lakes",
        )
        reader = shpreader.Reader(shpfile)
        segments = []
        e_range = EASTING_MAX - EASTING_MIN
        n_range = NORTHING_MAX - NORTHING_MIN
        for record in reader.records():
            if record.attributes.get("admin") != "United States of America":
                continue
            geom = record.geometry
            if geom.geom_type == "Polygon":
                polys = [geom]
            elif geom.geom_type == "MultiPolygon":
                polys = list(geom.geoms)
            else:
                continue
            for poly in polys:
                lons, lats = np.array(poly.exterior.coords).T
                ex, ny = proj(lons, lats)
                nx = (np.array(ex) - EASTING_MIN) / e_range
                ny_norm = (np.array(ny) - NORTHING_MIN) / n_range
                segments.append((nx, ny_norm))
        _STATE_CACHE = segments

    for nx, ny in _STATE_CACHE:
        ax.plot(nx, ny, color=color, linewidth=linewidth, alpha=alpha)

In [ ]:
# --- Load generated tracks ---
gen_df = pd.read_parquet(SAMPLES_PATH)
print(f"Generated tracks: {len(gen_df):,} rows, "
      f"{gen_df['date'].nunique()} days, "
      f"{gen_df['realization_id'].max() + 1} realizations/day")

In [ ]:
# --- Load GT tracks + weather ---
years = SPLIT_YEARS[SPLIT]
all_files = sorted(
    f for f in os.listdir(RAW_DIR)
    if (f.endswith(".pt") or f.endswith(".pt.gz")) and int(f[:4]) in years
)
print(f"Loading {len(all_files)} days from {SPLIT} split...")

gt_rows = []
day_rows = []
for fname in all_files:
    sample = _load_pt(os.path.join(RAW_DIR, fname))
    date = sample["date"]
    tracks_raw = sample["tracks"]  # (N, 6): se, sn, ee, en, width, ef
    tracks_bl = coords_to_bearing_length(tracks_raw)  # se, sn, bearing, length, width, ef
    n_tracks = tracks_bl.shape[0]
    p99_stp = float(sample["wx"][STP_CHANNEL].max())

    for i in range(n_tracks):
        t = tracks_bl[i]
        gt_rows.append({
            "date": date,
            "se": float(t[0]), "sn": float(t[1]),
            "bearing": float(t[2]), "length": float(t[3]),
            "width": float(t[4]), "ef": int(t[5]),
        })

    day_rows.append({
        "date": date,
        "gt_count": n_tracks,
        "p99_stp": p99_stp,
        "is_null": n_tracks == 0,
    })

gt_df = pd.DataFrame(gt_rows)
day_df = pd.DataFrame(day_rows)
print(f"GT tracks: {len(gt_df):,} across {len(day_df)} days "
      f"({day_df['is_null'].sum()} null days)")

# Merge generated count stats per day
gen_counts = gen_df.groupby(["date", "realization_id"]).size().reset_index(name="count")
# Include zero-count realizations for days that appear in day_df
n_realizations = gen_df["realization_id"].max() + 1
all_combos = pd.MultiIndex.from_product(
    [day_df["date"], range(n_realizations)], names=["date", "realization_id"]
).to_frame(index=False)
gen_counts = all_combos.merge(gen_counts, on=["date", "realization_id"], how="left")
gen_counts["count"] = gen_counts["count"].fillna(0).astype(int)

gen_day_stats = gen_counts.groupby("date")["count"].agg(
    mean_gen_count="mean", std_gen_count="std"
).reset_index()
day_df = day_df.merge(gen_day_stats, on="date", how="left")
day_df["std_gen_count"] = day_df["std_gen_count"].fillna(0)

In [ ]:
# --- STP deciles ---
day_df["stp_decile"] = pd.qcut(day_df["p99_stp"], 10, labels=False, duplicates="drop")

# Attach stp_decile to gen_counts for later use
gen_counts = gen_counts.merge(day_df[["date", "stp_decile", "is_null"]], on="date", how="left")

print("STP decile ranges:")
for d in sorted(day_df["stp_decile"].unique()):
    sub = day_df[day_df["stp_decile"] == d]
    print(f"  Decile {d}: STP [{sub['p99_stp'].min():.3f}, {sub['p99_stp'].max():.3f}] "
          f"({len(sub)} days)")

## Count Distribution Analysis

In [ ]:
# --- Count histograms: GT vs Generated ---
gt_day_counts = day_df["gt_count"].values
gen_real_counts = gen_counts["count"].values

max_count = max(gt_day_counts.max(), gen_real_counts.max())
bins = np.arange(-0.5, max_count + 1.5, 1)

ks_stat, ks_p = stats.ks_2samp(gt_day_counts, gen_real_counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gt_day_counts, bins=bins, density=True, alpha=0.6, color="C0",
        label=f"Ground Truth (N={len(gt_day_counts)} days)")
ax.hist(gen_real_counts, bins=bins, density=True, alpha=0.6, color="C1",
        label=f"Generated (N={len(gen_real_counts):,} realizations)")
ax.set_xlabel("Tornado count per day/realization")
ax.set_ylabel("Density")
ax.set_title("Daily Tornado Count Distribution")
ax.legend()
ax.annotate(f"KS = {ks_stat:.3f}, p = {ks_p:.3g}",
            xy=(0.97, 0.95), xycoords="axes fraction",
            ha="right", va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8))
fig.tight_layout()
plt.show()

In [ ]:
# --- Per-day count scatter with ±1σ ---
fig, ax = plt.subplots(figsize=(6, 6))
ax.errorbar(
    day_df["gt_count"], day_df["mean_gen_count"],
    yerr=day_df["std_gen_count"],
    fmt="o", markersize=3, alpha=0.5, elinewidth=0.5, capsize=0, color="C0",
)
lim = max(day_df["gt_count"].max(), day_df["mean_gen_count"].max()) + 1
ax.plot([0, lim], [0, lim], "--", color="gray", linewidth=0.8, label="1:1")
ax.set_xlim(-0.5, lim)
ax.set_ylim(-0.5, lim)
ax.set_xlabel("GT tornado count")
ax.set_ylabel("Mean generated count")
ax.set_title("Per-Day Count: GT vs Generated (±1σ)")
ax.legend()
ax.set_aspect("equal")
fig.tight_layout()
plt.show()